In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici")
dbutils.widgets.text("esquema", "bronze")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

In [0]:
ecobici_schema = StructType([
    StructField("anio", IntegerType(), True),
    StructField("mes", StringType(), True),
    StructField("fecha", DateType(), True),
    StructField("genero", StringType(), True),
    StructField("viajes", IntegerType(), True)
])


In [0]:
df = spark.read.format("csv") \
    .option("header", True) \
    .schema(ecobici_schema) \
    .load("/Volumes/catalog_ecobici/raw/raw_files/ecobici2024.csv")

In [0]:
df_final = df.withColumn("ingestion_date", current_timestamp())

In [0]:
df_final.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema}.ecobici")

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM catalog_ecobici.bronze.ecobici;

In [0]:
%sql
SELECT *
FROM catalog_ecobici.bronze.ecobici
LIMIT 10;

In [0]:
%sql
SELECT 
    MIN(fecha) AS fecha_inicio,
    MAX(fecha) AS fecha_fin
FROM catalog_ecobici.bronze.ecobici;

In [0]:
%sql
SELECT fecha, genero, COUNT(*) AS duplicados
FROM catalog_ecobici.bronze.ecobici
GROUP BY fecha, genero
HAVING COUNT(*) > 1;